# Trust-Aware Pneumonia Detection using Explainable AI

This notebook demonstrates a **Trust-Aware Pneumonia Detection** pipeline using deep learning and Explainable AI. The model classifies chest X-ray images as **Normal** or **Pneumonia** and explains the prediction using **Grad-CAM heatmaps**.

The project focuses on improving transparency in medical AI by combining:
- Pneumonia classification
- Confidence score
- Grad-CAM based visual explanation
- Trust score estimation

> Note: This notebook is designed for educational and open-source contribution purposes. It is not intended for real clinical diagnosis.

## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

print('TensorFlow version:', tf.__version__)

## 2. Dataset Path

Use the Kaggle **Chest X-Ray Images (Pneumonia)** dataset.

Expected folder structure:

```text
chest_xray/
│
├── train/
│   ├── NORMAL/
│   └── PNEUMONIA/
│
├── val/
│   ├── NORMAL/
│   └── PNEUMONIA/
│
└── test/
    ├── NORMAL/
    └── PNEUMONIA/
```

Update `DATASET_PATH` according to your local system.

In [ ]:
DATASET_PATH = 'chest_xray'

TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR = os.path.join(DATASET_PATH, 'val')
TEST_DIR = os.path.join(DATASET_PATH, 'test')

IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 5

print('Train directory:', TRAIN_DIR)
print('Validation directory:', VAL_DIR)
print('Test directory:', TEST_DIR)

## 3. Data Preprocessing

The images are resized to `224 × 224` and preprocessed using the ResNet50 preprocessing function. Data augmentation is applied to the training set to improve generalization.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=1,
    class_mode='binary',
    shuffle=False
)

print('Class indices:', train_generator.class_indices)

## 4. Build Transfer Learning Model

ResNet50 is used as a feature extractor. A custom classification head is added for binary pneumonia classification.

In [ ]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model

For quick demonstration, the model is trained for a small number of epochs. Increase `EPOCHS` for better performance.

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS
)

## 6. Plot Training Performance

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

## 7. Evaluate the Model

In [ ]:
predictions = model.predict(test_generator)
y_pred = (predictions > 0.5).astype(int).flatten()
y_true = test_generator.classes

print('Accuracy:', accuracy_score(y_true, y_pred))
print('\nClassification Report:\n')
print(classification_report(y_true, y_pred, target_names=list(test_generator.class_indices.keys())))
print('\nConfusion Matrix:\n')
print(confusion_matrix(y_true, y_pred))

## 8. Grad-CAM Explainability

Grad-CAM highlights the image regions that influenced the model prediction. This makes the result more interpretable by showing whether the model focuses on meaningful lung regions.

In [ ]:
def get_gradcam_heatmap(img_array, model, last_conv_layer_name='conv5_block3_out'):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()


def display_gradcam(image_path, model):
    img = tf.keras.preprocessing.image.load_img(image_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    original_img = img_array.astype('uint8')

    input_array = np.expand_dims(img_array, axis=0)
    input_array = preprocess_input(input_array)

    prediction = model.predict(input_array)[0][0]
    predicted_label = 'PNEUMONIA' if prediction > 0.5 else 'NORMAL'
    confidence = prediction if prediction > 0.5 else 1 - prediction

    heatmap = get_gradcam_heatmap(input_array, model)
    heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    heatmap = np.uint8(255 * heatmap)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

    superimposed_img = cv2.addWeighted(original_img, 0.6, heatmap, 0.4, 0)

    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    plt.imshow(original_img)
    plt.title('Original X-ray')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(superimposed_img)
    plt.title(f'Grad-CAM: {predicted_label} ({confidence:.2f})')
    plt.axis('off')
    plt.show()

    return predicted_label, float(confidence), heatmap

## 9. Trust Score Calculation

The trust score combines model confidence and explanation strength. In a real clinical-grade system, this can be expanded using localization quality and heatmap stability.

In [ ]:
def calculate_trust_score(confidence, heatmap):
    explanation_strength = np.mean(heatmap) / 255.0
    trust_score = (0.7 * confidence) + (0.3 * explanation_strength)
    return round(float(trust_score), 3)


def get_trust_level(trust_score):
    if trust_score >= 0.80:
        return 'High Trust'
    elif trust_score >= 0.60:
        return 'Medium Trust'
    else:
        return 'Low Trust'

## 10. Test on a Single Image

Update `sample_image_path` with any image from the test folder.

In [ ]:
sample_image_path = os.path.join(TEST_DIR, 'PNEUMONIA', os.listdir(os.path.join(TEST_DIR, 'PNEUMONIA'))[0])

predicted_label, confidence, heatmap = display_gradcam(sample_image_path, model)
trust_score = calculate_trust_score(confidence, heatmap)
trust_level = get_trust_level(trust_score)

print('Prediction:', predicted_label)
print('Confidence:', round(confidence, 3))
print('Trust Score:', trust_score)
print('Trust Level:', trust_level)

## 11. Save Model

The trained model can be saved for future inference or deployment.

In [ ]:
model.save('trust_aware_pneumonia_model.h5')
print('Model saved successfully.')

## 12. Conclusion

This project demonstrates how Explainable AI can improve transparency in medical image classification. Instead of only predicting whether pneumonia is present, the notebook also shows **why** the model made the prediction using Grad-CAM.

The trust-aware approach helps users interpret model outputs more carefully by combining prediction confidence with visual explanation strength.

### Future Scope
- Add SHAP or LIME explanations
- Add lung segmentation based localization score
- Add heatmap stability under image perturbations
- Deploy as a Streamlit web application
- Improve trust score calibration using expert annotations